In [469]:
import cv2
import mediapipe as mp
import numpy as np

print("OpenCV:", cv2.__version__)
print("MediaPipe:", mp.__version__)
print("NumPy:", np.__version__)

OpenCV: 5.0.0
MediaPipe: 0.10.9
NumPy: 2.4.6


In [470]:
mppose = mp.solutions.pose
mpdraw = mp.solutions.drawing_utils

In [471]:
data = mppose.Pose()

In [472]:
bowling_hand=input("Left or right")

In [473]:
elbow_angles=[]

In [474]:
import math

bowling_hand = bowling_hand.lower()

f=0
all_frame_data = []
elbow_angles = []
start_found = False
release_found=False
start_frame = None
start_angle = None


video = cv2.VideoCapture("videos/l1.mp4")
while True:
    suc, img = video.read()
    frame_data = []
    if not suc:
        break

    img1 = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    result = data.process(img1)

    if result.pose_landmarks:
        f += 1
        landmarks = result.pose_landmarks.landmark

        if bowling_hand == "r":
            selected_landmarks = [
                mppose.PoseLandmark.RIGHT_SHOULDER,
                mppose.PoseLandmark.RIGHT_ELBOW,
                mppose.PoseLandmark.RIGHT_WRIST
            ]
        else:
            selected_landmarks = [
                mppose.PoseLandmark.LEFT_SHOULDER,
                mppose.PoseLandmark.LEFT_ELBOW,
                mppose.PoseLandmark.LEFT_WRIST
            ]

        for landmark_id in selected_landmarks:
            landmark = landmarks[landmark_id]
            frame_data.append([
                f,
                landmark_id,
                landmark.x,
                landmark.y,
                landmark.z,
                landmark.visibility
            ])

        
        all_frame_data.append(frame_data)



        mpdraw.draw_landmarks(
            img,
            result.pose_landmarks,
            mppose.POSE_CONNECTIONS
        )

        

    cv2.imshow("img", img)

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

video.release()
cv2.destroyAllWindows()


In [475]:
best_horizontal=1
best_h_wrist=1

best_release=1
best_e_w=1

start_index = None
release_index = None

start_found=False

for index,frame in enumerate(all_frame_data):
    shoulder=frame[0]
    elbow=frame[1]
    wrist=frame[2]
    if elbow[2]<shoulder[2] and wrist[3]<shoulder[3]:
        if abs(shoulder[3]-elbow[3])<best_horizontal and abs(elbow[3]-wrist[3])<best_h_wrist:
            best_horizontal=abs(shoulder[3]-elbow[3])
            best_h_wrist=abs(elbow[3]-wrist[3])

            best_h_start=[shoulder[3],elbow[3],wrist[3],shoulder[3],elbow[3],wrist[3]]

            best_start_frame=frame
            start_index=index
            start_found=True
            

if start_found is True:
    for i in range(start_index+1,len(all_frame_data)):
        frame=all_frame_data[i]
        shoulder=frame[0]
        elbow=frame[1]
        wrist=frame[2]
        if start_index is not None:
         if elbow[3]<shoulder[3]:
            if abs(shoulder[2]-elbow[2])<best_release and abs(elbow[2]-wrist[2])<best_e_w:
                best_release=abs(shoulder[2]-elbow[2])
                best_e_w=abs(elbow[2]-wrist[2])

                best_r=[shoulder[2],elbow[2],wrist[2]]

                best_release_frame=frame
                release_index=i
                
                


In [476]:
print(best_h_start)
print(best_r)

[0.305054634809494, 0.31483983993530273, 0.2920137941837311, 0.305054634809494, 0.31483983993530273, 0.2920137941837311]
[0.5198409557342529, 0.5140942335128784, 0.4835950434207916]


In [477]:
print(start_index)
print(release_index)

235
267


In [478]:
window=all_frame_data[start_index:release_index+1]

In [479]:
elbow_angles = []

import math

for f in window:

    shoulder = f[0]
    elbow = f[1]
    wrist = f[2]

    sx, sy = shoulder[2], shoulder[3]
    ex, ey = elbow[2], elbow[3]
    wx, wy = wrist[2], wrist[3]

    # Shoulder -> Elbow
    a = (
        sx - ex,
        sy - ey
    )

    # Wrist -> Elbow
    b = (
        wx - ex,
        wy - ey
    )

    dot = (
        a[0] * b[0] +
        a[1] * b[1]
    )

    mag_a = math.sqrt(
        a[0]**2 +
        a[1]**2
    )

    mag_b = math.sqrt(
        b[0]**2 +
        b[1]**2
    )

    if mag_a != 0 and mag_b != 0:

        value = dot / (mag_a * mag_b)

        # Prevent acos domain error
        value = max(-1, min(1, value))

        angle = math.degrees(
            math.acos(value)
        )

        elbow_angles.append(angle)

        print(
            "Frame:", f[0][0],
            "Angle:", angle
        )

Frame: 236 Angle: 157.04572779029425
Frame: 237 Angle: 151.7906053769152
Frame: 238 Angle: 153.05401736286473
Frame: 239 Angle: 150.48706649674452
Frame: 240 Angle: 152.58915824447863
Frame: 241 Angle: 152.39152264517884
Frame: 242 Angle: 147.62466793186084
Frame: 243 Angle: 147.43011880957783
Frame: 244 Angle: 149.05841875641175
Frame: 245 Angle: 149.11569187199512
Frame: 246 Angle: 155.04571704514876
Frame: 247 Angle: 155.9330163544192
Frame: 248 Angle: 155.74708284932566
Frame: 249 Angle: 157.65555151091513
Frame: 250 Angle: 158.43210372286782
Frame: 251 Angle: 164.72426413193514
Frame: 252 Angle: 167.59551343684137
Frame: 253 Angle: 168.98770010851078
Frame: 254 Angle: 173.90749852984854
Frame: 255 Angle: 175.51980311282222
Frame: 256 Angle: 174.90179565107965
Frame: 257 Angle: 175.78168909779944
Frame: 258 Angle: 179.50546547324987
Frame: 259 Angle: 179.55110810482824
Frame: 260 Angle: 179.5107702377733
Frame: 261 Angle: 177.28854810187738
Frame: 262 Angle: 177.8514352907418
Frame

In [480]:
print(elbow_angles)

[157.04572779029425, 151.7906053769152, 153.05401736286473, 150.48706649674452, 152.58915824447863, 152.39152264517884, 147.62466793186084, 147.43011880957783, 149.05841875641175, 149.11569187199512, 155.04571704514876, 155.9330163544192, 155.74708284932566, 157.65555151091513, 158.43210372286782, 164.72426413193514, 167.59551343684137, 168.98770010851078, 173.90749852984854, 175.51980311282222, 174.90179565107965, 175.78168909779944, 179.50546547324987, 179.55110810482824, 179.5107702377733, 177.28854810187738, 177.8514352907418, 169.1237451834872, 169.2924861252372, 170.5153629655232, 166.7362607430085, 166.92801926882746, 166.4299835205504]


In [481]:
print("MAX ANGLE:", max(elbow_angles))
print("MIN ANGLE:", min(elbow_angles))
print("EXTENSION:",max(elbow_angles)-min(elbow_angles))

MAX ANGLE: 179.55110810482824
MIN ANGLE: 147.43011880957783
EXTENSION: 32.12098929525041
